* Student: Diego Bernal
* Id:1257853

### Report

## Introduction

This project aims to use the knowledge learned during the PyTorch course to solve different challenges in image analysis, starting with basic tasks like classification and moving toward more complex and real-world applications.

In the first stage, we worked with the Fruit Image Dataset: Fresh and Rotten Fruits, which contains images of fruits divided into six categories: apples, bananas, and oranges — both fresh and rotten. The dataset is organized into separate folders for training and testing. Using this dataset, we trained a convolutional neural network (CNN) from scratch to build a model that can tell the difference between the different fruit classes. This part of the project included data preprocessing, model design, training, evaluation, and showing the results.

Later, to go deeper into image analysis and use more advanced techniques from the course, we used a pretrained model based on EfficientNet. This model was fine-tuned to classify the same six fruit categories. We also added the ability to analyze external images and used Grad-CAM, a visualization tool that helps us understand which parts of the image the model focused on when making its predictions.

To take things further, we implemented a DETR (DEtection TRansformer) model, supported by a ResNet-50 backbone, trained on the COCO dataset classes. This model helped us not only recognize objects but also locate them within an image.

With that knowledge, we applied the model to object detection in videos, which is a big step toward more dynamic and practical real-world applications.

Finally, we discussed a possible real-life use: integrating these models into computer vision systems for robots or machines. This would allow them to understand their surroundings using real-time images. This kind of visual analysis could be used in automation, agriculture, manufacturing, and many other fields.

## Part 1: Basic Model

* dataset:Fruit Image Dataset: Fresh and Rotten Fruits
* model: CNN

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report
import numpy as np

### - Data preprocessing

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5])
])

test_transforms = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5])
])


dataset_root = '../data'


train_dataset_full = datasets.ImageFolder(root=f'{dataset_root}/train', transform=train_transforms)
test_dataset = datasets.ImageFolder(root=f'{dataset_root}/test', transform=test_transforms)


val_size = int(0.1 * len(train_dataset_full))
train_size = len(train_dataset_full) - val_size
train_dataset, val_dataset = random_split(train_dataset_full, [train_size, val_size])
val_dataset.dataset.transform = test_transforms  # sin data augmentation en validación


batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

class_names = train_dataset_full.classes
print("Clases:", class_names)

### - Model selection (CNN)

In [ ]:
class FruitClassifierCNN(nn.Module):
    def __init__(self, num_classes):
        super(FruitClassifierCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.fc = nn.Sequential(
            nn.Linear(128 * 16 * 16, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

num_classes = len(class_names)
model = FruitClassifierCNN(num_classes)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

### - Training

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, device, num_epochs=10, patience=3):
    best_model_wts = copy.deepcopy(model.state_dict())
    best_val_loss = float('inf')
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    patience_counter = 0

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            
        train_loss = running_loss / len(train_loader.dataset)
        train_acc = correct / len(train_loader.dataset)
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        
        # Evaluación en validación
        model.eval()
        val_loss = 0.0
        val_correct = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                val_correct += (outputs.argmax(1) == labels).sum().item()
        val_loss /= len(val_loader.dataset)
        val_acc = val_correct / len(val_loader.dataset)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        print(f"Epoch {epoch+1}/{num_epochs}: Train loss={train_loss:.4f}, Train Acc={train_acc:.4f}, Val loss={val_loss:.4f}, Val Acc={val_acc:.4f}")
        
        # Early Stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping activado")
                break

    model.load_state_dict(best_model_wts)
    return model, history

### - Hyperparameter tuning

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

learning_rates = [0.001, 0.0005]
batch_sizes = [16, 32]
num_epochs = 10 

results = {}  

for lr in learning_rates:
    for bs in batch_sizes:
        print(f"\nLearning rate = {lr} and batch size = {bs}")

        train_loader = DataLoader(train_dataset, batch_size=bs, shuffle=True)
        val_loader   = DataLoader(val_dataset, batch_size=bs, shuffle=False)
        
        model = FruitClassifierCNN(num_classes)
        model.to(device)
        
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)
        
        inicio = time.time()
        model, history = train_model(model, train_loader, val_loader, criterion, optimizer, device, num_epochs=num_epochs, patience=3)
        duracion = time.time() - inicio
        
        best_val_acc = max(history['val_acc'])
        results[(lr, bs)] = {'history': history, 'best_val_acc': best_val_acc, 'training_time': duracion}
        
        print(f"Resultado para lr={lr}, batch_size={bs}: Mejor Val Acc = {best_val_acc:.4f}, Tiempo de entrenamiento = {duracion:.2f} s")

# results
print("\nSummary of Hyperparameter Tuning:")
for (lr, bs), result in results.items():
    print(f"Learning Rate: {lr}, Batch Size: {bs}, Mejor Val Acc: {result['best_val_acc']:.4f}, Tiempo: {result['training_time']:.2f} s")


### - Evaluation

In [ ]:
best_params = max(results, key=lambda x: results[x]['best_val_acc'])
print(f"\nMejor combinación de hiperparámetros: Learning Rate: {best_params[0]}, Batch Size: {best_params[1]}")

# Regenerar DataLoader con el mejor batch_size
train_loader = DataLoader(train_dataset, batch_size=best_params[1], shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=best_params[1], shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=best_params[1], shuffle=False)

# Instanciar y entrenar de nuevo el modelo con la mejor tasa de aprendizaje
model = FruitClassifierCNN(num_classes)
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=best_params[0])
model, history = train_model(model, train_loader, val_loader, criterion, optimizer, device, num_epochs=25, patience=5)

# Evaluar en el conjunto de Test
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        preds = outputs.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print("\nReport Clasification in Test:")
print(classification_report(all_labels, all_preds, target_names=class_names))

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Validation Loss')
plt.title('Loss per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)


plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='Train Accuracy')
plt.plot(history['val_acc'], label='Validation Accuracy')
plt.title('Accuracy per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
torch.save(model.state_dict(), 'modelBasic.pth')